# Forecasting the CLEAR beam charge — kickoff notebook

**Case owner:** Antonio Gilardi (CERN) · part of the [optimisation challenge](../)

CLEAR is an electron linac at CERN. A laser hits a photocathode in an RF gun, and the
charge of the electron bunch it produces is measured by `BCM_GUN`, shot after shot. That
charge drifts and jitters, and users need it stable.

This notebook works on real shot-by-shot data from October 2026. Every shot records the
charge plus everything that could drive it: the laser, the gun RF, klystron 15, a few
magnets, and the hall temperature and humidity. One question runs through it:

> **How much of the charge can the other variables explain, on the same shot and ahead of
> time?**

**This is one way through the challenge, not the answer.** Each step gives a simple
baseline and says what to look at. Change anything, replace anything, and build your own
pipeline. Only keep the common rules of section 1b, so that results stay comparable.

## Contents

| Section | Tier | What it does |
|---|---|---|
| 1. Setup and rules | | imports, and the common rules (split, dropouts, baseline) |
| 2. From raw data to a table | 🟢 Easy | open raw chunks, reduce them to one row per shot |
| 3. Understand and clean | 🟢 Easy | quality, dropouts, synchronisation, waveform windows, correlation study |
| 4. Over to you | 🟡 Medium, 🔴 Hard | the open tasks, how to evaluate them, the references to beat |
| 5. Next steps | | where to go from here |

**How to use it:** run the cells from top to bottom. Sections 2 and 3 work through the
Easy tier as one example; the Medium and Hard tiers (section 4) are left open for you.
**Your turn** boxes link to the goals in `objectives.md`.

**Words used throughout**

| Word | Meaning |
|---|---|
| shot | one laser pulse and the electron bunch it makes; one row of the table |
| chunk | one raw data file, holding 100 shots |
| stamp | the time a device says its reading was taken |
| dropout | a shot where the charge collapsed (blocked laser, missed shot, bad reading) |
| test region | the last 10 % of each run's shots, used only to evaluate |
| virtual diagnostic | inferring the charge from the other variables of the same shot |
| horizon `h` | how many shots ahead a forecast looks |

## 1. Setup and rules

### 1a. Imports

If an import fails, set up the environment first: run `./setup_venv.sh` in this folder, then
choose the kernel **Python (forecasting)** (or `pip install -r requirements.txt` yourself).

In [ ]:
# first check that every package is there, with a clear message if not
import importlib.util
NEEDED = {'numpy': 'numpy', 'pandas': 'pandas', 'pyarrow': 'pyarrow',   # module: pip name
          'matplotlib': 'matplotlib', 'scipy': 'scipy', 'sklearn': 'scikit-learn'}
missing = [pip for module, pip in NEEDED.items() if importlib.util.find_spec(module) is None]
if missing:
    raise ImportError(f'Missing packages: {", ".join(missing)}.\n'
                      'Run ./setup_venv.sh and choose the kernel "Python (forecasting)", '
                      f'or run in a cell:  %pip install {" ".join(missing)}  and restart the kernel.')

import glob
import os
import pickle
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.feature_selection import mutual_info_regression

SEED = 42                     # fixed seed, so every run of the notebook gives the same numbers
np.random.seed(SEED)
print('Imports OK')

### 1b. Common rules (the same for every team)

So that results can be compared between teams, everyone uses the same split, dropout
definition and baseline, written as the functions below. **Use them as they are.** In
short:

- **Test region:** the last `TEST_FRAC` (10 %) of each run's shots. `cut_shot` gives its
  first shot. Train only on shots before it; never shuffle.
- **Dropouts:** a charge below `DROP_FRAC` (30 %) of the median of the surrounding 201
  shots, or a missing charge. Dropouts are not evaluated.
- **Baseline:** the mean of the last `W` (9) good shots. This is the naive forecast.
- **Common reference metric:** MAE in nC on the test shots that are not dropouts (lower is
  better), with R² alongside. Forecasts also get `dMAE = MAE(baseline) − MAE(model)`:
  positive means you beat the rolling mean.

MAE is only a common reference. Try other metrics too, and explain what each one means
physically. Always check your approach for leakage from the test region.

In [ ]:
TARGET = 'BCM_GUN'    # the charge at the gun, in nC
TEST_FRAC = 0.10      # the last 10 % of a run's shots are the test region
DROP_FRAC = 0.3       # dropout: charge below 30 % of the local (201-shot) median
W = 9                 # window of the rolling-mean baseline, in shots


def cut_shot(df):
    # first shot of the test region
    return df.index[int(len(df) * (1 - TEST_FRAC))]


def dropouts(df):
    # True for shots whose charge collapsed or is missing
    q = df[TARGET]
    local_median = q.rolling(201, center=True, min_periods=20).median()
    return (q < DROP_FRAC * local_median) | q.isna()


def scored_shots(df):
    # test shots whose charge is a real measurement
    return df.index[(df.index >= cut_shot(df)) & ~dropouts(df)]


def rolling_baseline(df):
    # at each shot: the mean of the last W good shots (the naive forecast)
    return df[TARGET].where(~dropouts(df)).rolling(W, min_periods=1).mean()


def metrics(y, p):
    err = y - p
    return {'MAE': float(np.mean(np.abs(err))),
            'R2': float(1 - np.sum(err ** 2) / np.sum((y - y.mean()) ** 2))}


def score_vd(df, pred):
    # Virtual diagnostic. pred: a Series of predictions indexed by shot.
    # Returns MAE, R2, and the fraction of evaluated shots that got a prediction (coverage).
    shots = scored_shots(df)
    pred = pd.Series(pred).reindex(shots)
    ok = pred.notna()
    res = metrics(df.loc[shots[ok], TARGET].to_numpy(float), pred[ok].to_numpy(float))
    res['coverage'] = float(ok.mean())
    return res


def score_forecast(df, pred):
    # Forecasting. pred: a DataFrame with columns shot, h, pred, where `shot` is the shot
    # being predicted, from information up to shot - h.
    # Returns one row per horizon h, with the baseline at the same origins and
    # dMAE = MAE(baseline) - MAE(model).
    shots = scored_shots(df)
    base = rolling_baseline(df)
    rows = []
    for h, g in pd.DataFrame(pred).groupby('h'):
        h = int(h)
        p = g.set_index('shot')['pred'].reindex(shots)
        origin_ok = (shots - h) >= df.index[0]
        ok = p.notna() & origin_ok
        y = df.loc[shots[ok], TARGET].to_numpy(float)
        model = metrics(y, p[ok].to_numpy(float))
        b = metrics(y, base.reindex(shots[ok] - h).to_numpy(float))
        rows.append({'h': h, 'MAE': model['MAE'], 'R2': model['R2'],
                     'baseline_MAE': b['MAE'], 'dMAE': b['MAE'] - model['MAE'],
                     'coverage': float(ok.sum() / origin_ok.sum())})
    return pd.DataFrame(rows).set_index('h')

## 2. From raw data to a table

The raw data are big: a run is 50–125 GB of pickle files ("chunks" of 100 shots), almost
all of it camera images. Turning a run into a table with **one row per shot** is your first
job (Easy tier). This section does it for **one chunk**, to show how. Processing whole
runs, and improving the features, is up to you.

### 2a. Get the raw data

Download a run, or the first few chunks of one, from the CERNBox share (link in
`data/README.md`) into `data/raw/<run>/`:

```
data/raw/acquisition_2026-10-09_07-46-07/
├── parameters.pkl        the device addresses that were read
└── chunk_00000.pkl ...   100 shots each
```

One chunk is about 0.5 GB. Start with one or two chunks, and download whole runs once your
pipeline works. Git ignores `data/raw/`.

**How a shot is recorded.** The acquisition notices a new shot when `BCM_GUN`'s stamp
changes, then reads all 23 parameters **in parallel**. Each reading keeps its own stamp, so
a device that had not updated yet hands over the *previous* shot's value. Putting every
reading back on its own shot is part of section 3.

In [ ]:
raw_runs = sorted(d for d in glob.glob('data/raw/acquisition_*')
                  if glob.glob(os.path.join(d, 'chunk_*.pkl')))
if not raw_runs:
    raise FileNotFoundError('No raw data in data/raw/. Download at least one chunk of a run from '
                            'the CERNBox share into data/raw/<run>/ (see data/README.md).')
print('runs with raw chunks:')
for i, r in enumerate(raw_runs):
    print(f"  [{i}] {os.path.basename(r)}: {len(glob.glob(os.path.join(r, 'chunk_*.pkl')))} chunk(s)")

RAW_RUN = raw_runs[-1]        # <- change this to pick another run, e.g. raw_runs[0]
run_name = os.path.basename(RAW_RUN)
chunk_files = sorted(glob.glob(os.path.join(RAW_RUN, 'chunk_*.pkl')))
print(f'\nusing {run_name}')

**Opening a chunk safely.** A chunk is a Python pickle. A plain `pickle.load` can run code
hidden in the file, and it would also need CERN's `pyda` package to rebuild control-system
objects. `load_chunk` below rebuilds only numpy arrays, datetimes and plain Python types.
Anything else becomes an inert `Opaque` placeholder, so no `pyda` is needed and nothing in
the file can run.

A chunk is a list of shot dictionaries:
`{'shot', 'pc_time', 'trigger_stamp', <NAME>: value, <NAME>_stamp: stamp, ...}`.

In [ ]:
SAFE = {   # the only types a chunk may rebuild
    'builtins': {'set', 'frozenset', 'dict', 'list', 'tuple', 'bytearray', 'bytes',
                 'complex', 'int', 'float', 'str', 'bool', 'slice', 'range', 'object'},
    'datetime': {'datetime', 'date', 'time', 'timedelta', 'timezone'},
    'collections': {'OrderedDict', 'defaultdict'},
    'copyreg': {'_reconstructor'},
    '_codecs': {'encode'},
    'numpy': {'ndarray', 'dtype', 'bool_'},
    'numpy.core.multiarray': {'_reconstruct', 'scalar'},
    'numpy._core.multiarray': {'_reconstruct', 'scalar'},
    'numpy.core.numeric': {'_frombuffer'},
    'numpy._core.numeric': {'_frombuffer'},
}


class Opaque:
    # placeholder for any object outside SAFE; keeps its arguments and state for inspection
    def __new__(cls, *args, **kwargs):
        return object.__new__(cls)

    def __init__(self, *args, **kwargs):
        self.args = args

    def __setstate__(self, state):
        self.state = state

    def __repr__(self):
        return f'<{type(self).__module__}.{type(self).__name__}>'


class SafeUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if name in SAFE.get(module, ()):
            if module.startswith('numpy.core') and hasattr(np, '_core'):
                module = 'numpy._core' + module[len('numpy.core'):]   # renamed in numpy 2
            return super().find_class(module, name)
        return type(name, (Opaque,), {'__module__': module})


def load_chunk(path):
    # a raw chunk file -> list of shot dictionaries
    with open(path, 'rb') as f:
        return SafeUnpickler(f).load()


def data_array(v):
    # The measurement inside a value: a number, a waveform (1-D array) or an image (2-D).
    # Whole device properties (dicts) keep it under 'image2D' (camera), 'signal' (scope)
    # or 'samples' (RF loop, klystron); their other keys are metadata.
    if isinstance(v, dict):
        v = next((v[k] for k in ('image2D', 'signal', 'samples') if k in v), None)
    if v is None or isinstance(v, (int, float)):
        return v
    a = np.squeeze(np.asarray(v))
    return a if a.dtype.kind in 'iufb' else None

### 2b. Look at one shot

Before reducing anything, look at what a shot holds.

In [ ]:
chunk = load_chunk(chunk_files[0])
first = chunk[0]
print(f'{os.path.basename(chunk_files[0])}: {len(chunk)} shots. What one shot holds:\n')
print(f'  {"key":16s} {"type":9s} {"shape":14s} own stamp')
for k, v in first.items():
    if k.endswith('_stamp'):
        continue
    a = data_array(v)
    has_stamp = 'yes' if k + '_stamp' in first else ''
    print(f'  {k:16s} {type(v).__name__:9s} {str(getattr(a, "shape", "")):14s} {has_stamp}')
print('\nA whole device property, e.g. the camera BTV0125, holds these fields:')
for k, v in first['BTV0125'].items():
    print(f"  {k:30s} {type(v).__name__}")

In [ ]:
shot = chunk[len(chunk) // 2]                              # one shot from the middle
waves = [k for k, v in shot.items()
         if not k.endswith('_stamp') and getattr(data_array(v), 'ndim', 0) == 1]
fig, axes = plt.subplots(2, 1 + (len(waves) + 1) // 2, figsize=(18, 6.5), squeeze=False)

# the laser spot on the virtual cathode, with its centroid
ax = axes[0, 0]
img = data_array(shot.get('BTV0125'))
if getattr(img, 'ndim', 0) == 2:
    lo, hi = np.percentile(img, [1, 99.9])
    ax.imshow(img, cmap='inferno', vmin=lo, vmax=hi, origin='lower')
    signal = np.clip(img.astype(float) - np.median(img), 0, None)   # background removed
    cy, cx = (np.indices(img.shape) * signal).sum(axis=(1, 2)) / signal.sum()
    ax.plot(cx, cy, 'c+', ms=15)
    ax.set_title(f"BTV0125, shot {shot['shot']} (+ = centroid)", fontsize=9)

# the charge over the chunk
ax = axes[1, 0]
ax.plot([s[TARGET] for s in chunk], '.-')
ax.set_title(f'{TARGET} over this chunk [nC]', fontsize=9)
ax.set_xlabel('shot in chunk')

# every waveform, about 20 shots overlaid
for ax, name in zip(axes[:, 1:].T.ravel(), waves):
    for s in chunk[::max(1, len(chunk) // 20)]:
        a = data_array(s.get(name))
        if getattr(a, 'ndim', 0) == 1:
            t = s[name].get('times') if isinstance(s[name], dict) else None
            x = t if t is not None and len(t) == len(a) else np.arange(len(a))
            ax.plot(x, a, lw=0.6, alpha=0.5)
    ax.set_title(name, fontsize=9)
for ax in axes[:, 1:].ravel()[len(waves):]:
    ax.set_visible(False)
for ax in axes.ravel():
    ax.tick_params(labelsize=7)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**What to look for:** does each waveform show a clear pulse, or only noise? Is a trace
flat-topped, which would mean the digitiser is saturated? A signal that never changes with
the charge cannot explain it, however good the model.

### 2c. Reduce the run to a table

Each shot becomes one row of numbers. This is one simple choice of features, not the
answer:

| raw value | columns in the table |
|---|---|
| scalar (charge, laser energy, magnet current, temperature, ...) | `<NAME>` |
| waveform (RF, klystron, laser scope) | `<NAME>.peak`, `.peak_pos`, `.integral`, `.baseline`, and `.w00` … `.w47`: the trace averaged in 48 windows |
| image (`BTV0125`, the laser spot on the virtual cathode) | `<NAME>.intensity`, `.cx`, `.cy` (centroid, px), `.sx`, `.sy` (rms size, px) |
| every parameter | `<NAME>.stamp_offset`: its stamp minus the shot's stamp, in s |
| device status | e.g. `BTV0125.isGoodExposure`, `laser_scope.saturated`, `PGLA.hotspot0` |

Nothing is synchronised, cleaned or removed here: every shot is kept as recorded. That
comes in section 3.

The camera images make this slow, so `MAX_CHUNKS` limits how many chunks are reduced. The
table is saved to `data/tables/`, so the next time it is simply loaded.

In [ ]:
N_WINDOWS = 48    # each waveform is also kept as 48 window averages (used in section 3d)
NOISE_CUT = 3     # camera pixels below 3 x the noise level count as background
SKIP = {'shot', 'pc_time', 'trigger_stamp', 'stale_params'}
STATUS = {        # device fields kept as status columns
    'BTV0125': ('isGoodExposure', 'acqImageNumberDiffFromLast'),
    'laser_scope': ('saturated', 'acquired', 'vertOffset_V', 'vertGain_VperLSB', 'horizOffset_ns'),
}
DEVICE_STAMPS = {'BTV0125': ('cycleStampGlob', 'imageTimeStamp')}   # the camera's own timing


def to_seconds(stamp):
    # a stamp (datetime, or a number in ns, us, ms or s) -> seconds since 1970
    if stamp is None:
        return np.nan
    if hasattr(stamp, 'timestamp'):
        return stamp.timestamp()
    try:
        x = float(stamp)
    except (TypeError, ValueError):
        return np.nan
    for limit, scale in ((1e17, 1e9), (1e14, 1e6), (1e11, 1e3)):
        if x > limit:
            return x / scale
    return x


def flag_value(v):
    # A status field as a number. CERN Boolean and enum objects arrive as Opaque
    # placeholders that keep their value in their state.
    if isinstance(v, Opaque):
        state = getattr(v, 'state', None)
        v = state.get('value') if isinstance(state, dict) else None
    try:
        return float(v)
    except (TypeError, ValueError):
        return np.nan


def waveform_features(a):
    a = np.asarray(a, float)
    baseline = a[:max(5, len(a) // 20)].mean()       # the start of the trace, before the pulse
    b = a - baseline
    i = int(np.argmax(np.abs(b)))
    feats = {'peak': b[i], 'peak_pos': float(i), 'integral': b.sum(), 'baseline': baseline}
    for k, part in enumerate(np.array_split(a, N_WINDOWS)):
        feats[f'w{k:02d}'] = part.mean()
    return feats


def image_features(img):
    img = np.asarray(img, float)
    background = np.median(img)
    noise = 1.4826 * np.median(np.abs(img - background))     # robust noise level
    img = img - background
    img[img < NOISE_CUT * max(noise, 1e-12)] = 0              # keep only the laser spot
    total = img.sum()
    if total <= 0:
        return dict.fromkeys(('intensity', 'cx', 'cy', 'sx', 'sy'), np.nan)
    py, px = img.sum(1), img.sum(0)
    x, y = np.arange(len(px)), np.arange(len(py))
    cx, cy = (px * x).sum() / total, (py * y).sum() / total
    return {'intensity': total, 'cx': cx, 'cy': cy,
            'sx': np.sqrt((px * (x - cx) ** 2).sum() / total),
            'sy': np.sqrt((py * (y - cy) ** 2).sum() / total)}


def device_status(name, d, t):
    out = {f: flag_value(d[f]) for f in STATUS.get(name, ()) if f in d}
    for f in DEVICE_STAMPS.get(name, ()):
        if f in d:
            offset = to_seconds(d[f]) - t
            if abs(offset) < 86400:                  # a real time stamp, not a counter
                out[f'{f}_offset'] = offset
    if 'hotSpotsSamples' in d:                       # RF loop and klystron: two reference samples
        for k, x in enumerate(np.ravel(np.asarray(d['hotSpotsSamples'], float))[:2]):
            out[f'hotspot{k}'] = x
    return out


def reduce_shot(shot):
    # one raw shot -> one flat row of numbers
    t = to_seconds(shot['trigger_stamp'])
    row = {'shot': shot['shot'], 't': t, 'pc_time': shot.get('pc_time')}
    for name, v in shot.items():
        if name in SKIP or name.endswith('_stamp'):
            continue
        row[f'{name}.stamp_offset'] = to_seconds(shot.get(name + '_stamp')) - t
        a = data_array(v)
        if a is not None and np.ndim(a) == 0:
            row[name] = float(a)
        elif a is not None and a.ndim == 1:
            row.update({f'{name}.{k}': x for k, x in waveform_features(a).items()})
        elif a is not None and a.ndim == 2:
            row.update({f'{name}.{k}': x for k, x in image_features(a).items()})
        if isinstance(v, dict):
            row.update({f'{name}.{k}': x for k, x in device_status(name, v, t).items()})
    return row


def is_window(c):
    # True for the waveform window columns <NAME>.w00 ... <NAME>.w47
    tail = c.rsplit('.', 1)[-1]
    return tail[0] == 'w' and tail[1:].isdigit()


def reduce_run(files):
    rows = []
    for i, path in enumerate(files):
        print(f'\r  reducing chunk {i + 1}/{len(files)}', end='', flush=True)
        rows += [reduce_shot(s) for s in load_chunk(path)]
    print()
    table = pd.DataFrame(rows).set_index('shot').sort_index()
    windows = [c for c in table.columns if is_window(c)]
    table[windows] = table[windows].astype(np.float32)        # half the memory, enough precision
    return table

In [ ]:
MAX_CHUNKS = 1    # <- None reduces every downloaded chunk of the run

files = chunk_files[:MAX_CHUNKS]
table_file = f'data/tables/{run_name}_{len(files)}chunks.parquet'
if os.path.exists(table_file):
    print(f'{table_file} already exists, so it is just loaded below (delete it to reduce again)')
else:
    start = time.time()
    table = reduce_run(files)
    os.makedirs('data/tables', exist_ok=True)
    table.to_parquet(table_file)
    print(f'{len(table)} shots -> {table_file} ({os.path.getsize(table_file) / 1e6:.1f} MB), '
          f'{time.time() - start:.0f} s, mostly spent on the camera images')

### 2d. Load the table

The runs are **not alike**: some are stable, some saturated, some single-bunch and some
multi-bunch, and in some the laser position or intensity was varied. None of this is
labelled. Look at the plots before trusting any result: a correlation study on a saturated
run *should* find little.

In [ ]:
# `recorded`: the table exactly as recorded. The evaluation helpers always use this one.
recorded = pd.read_parquet(table_file)
recorded = recorded[~recorded.index.duplicated()].sort_index()

# the 23 parameters, and the columns that belong to each
params = sorted(c[:-len('.stamp_offset')] for c in recorded.columns if c.endswith('.stamp_offset'))
param_cols = {p: [c for c in recorded.columns if (c == p or c.startswith(p + '.'))
                  and not c.endswith('.stamp_offset')] for p in params}

dt = np.nanmedian(np.diff(recorded['t']))          # time between two shots, in s
print(f'{run_name}: {len(recorded)} shots, {len(params)} parameters, {recorded.shape[1]} columns')
print(f"duration {(recorded['t'].max() - recorded['t'].min()) / 3600:.2f} h, "
      f'one shot every {dt:.3f} s ({1 / dt:.3f} Hz)')
if len(recorded) < 2000:
    n_test = len(recorded) - int(len(recorded) * (1 - TEST_FRAC))
    print(f'\n!! Only {len(recorded)} shots: every cell runs, but the test region is just '
          f'{n_test} shots, so the results are a demonstration, not a result. '
          'Reduce more chunks (MAX_CHUNKS) for real numbers.\n')
print('\nparameter          columns')
for p in params:
    cols = [c for c in param_cols[p] if not is_window(c)]
    n_win = sum(map(is_window, param_cols[p]))
    print(f'  {p:16s} {", ".join(cols)}' + (f'  (+{n_win} windows)' if n_win else ''))

> **Your turn:** reduce every run you want to use, in full (`MAX_CHUNKS = None`, or the same
> functions in your own script, which you can leave running). Improve the features: which
> part of a waveform, which property of the laser spot, actually matters?

## 3. 🟢 Easy — understand and clean the data

### 3a. Quality: missing values, timing, stale readings

`stamp_offset` is the key column. About 0 means the reading belongs to this shot. About
minus one shot spacing means it is the **previous** shot's reading, handed over again
because the device had not updated yet ("stale"). Slow devices (temperature, humidity,
magnet currents) legitimately update far less often than once per shot.

In [ ]:
SLOW = ('TEMP', 'HUMID', 'SNH', 'SNI', 'DHG')     # devices that update less than once per shot

rows = []
for p in params:
    first_col = recorded[param_cols[p][0]] if param_cols[p] else pd.Series(np.nan, index=recorded.index)
    offset = recorded[f'{p}.stamp_offset']
    same_stamp_as_previous = (offset + recorded['t']).diff().abs() < 1e-6
    rows.append({
        'parameter': p,
        'missing_%': 100 * first_col.isna().mean(),
        'median_offset_s': np.nanmedian(offset),
        'offset_over_0.5s_%': 100 * (offset.abs() > 0.5).mean(),
        'stale_%': np.nan if p.startswith(SLOW) else 100 * same_stamp_as_previous.mean(),
        'slow': p.startswith(SLOW),
    })
quality = pd.DataFrame(rows).set_index('parameter')
quality.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3))
minutes = (recorded['t'] - recorded['t'].min()) / 60
ax.plot(minutes, recorded[TARGET], '.', ms=1.5)
ax.set_xlabel('time [min]')
ax.set_ylabel(f'{TARGET} [nC]')
ax.set_title(f'{TARGET} over the run')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**What to look for:** a high `offset_over_0.5s_%` or `stale_%` means that device often
reports another shot's value, so synchronisation (3c) matters for it. In the charge plot,
look for slow drifts, steps, and sudden drops: they hint at the run's regime.

### 3b. Dropouts and bad shots

Two kinds of shot get a flag. They are **flagged, not deleted**: a virtual diagnostic is
evaluated without them, but a forecaster may want to know that the last few shots were bad.

- **Charge dropouts:** the charge collapses (blocked laser, missed shot, bad reading). This
  is not "low charge" in the physics sense.
- **Device flags:** the camera's `isGoodExposure`, and the scope's `saturated` and
  `acquired`. On these shots the charge is real; only that device's reading is doubtful.
  They are left out of *training*, but still evaluated.

In [ ]:
recorded['is_dropout'] = dropouts(recorded)          # the rule from section 1b

device_bad = pd.Series(False, index=recorded.index)
for col, bad_value in (('BTV0125.isGoodExposure', 0),
                       ('laser_scope.saturated', 1),
                       ('laser_scope.acquired', 0)):
    if col in recorded:
        is_bad = recorded[col] == bad_value
        print(f'{col} == {bad_value}: {is_bad.sum()} shots')
        device_bad |= is_bad
recorded['is_bad_device'] = device_bad

print(f"\n{recorded['is_dropout'].sum()} charge dropouts "
      f"({100 * recorded['is_dropout'].mean():.2f} % of shots), "
      f"{device_bad.sum()} shots with a device flag")

### 3c. Synchronisation

Each fast parameter is moved **shot by shot** to the row it belongs to:
`round(stamp_offset / shot spacing)` rows away. A row left with no reading of its own
becomes NaN, because that device missed the shot. Slow parameters are forward-filled
instead, since they did not change between updates.

The result is a new table, `synced`. From here on, all modelling uses `synced`; scoring
still uses `recorded`, whose charge column is the same.

**Cross-check:** dropouts are a sharp marker. When the laser is blocked, both the charge
and the laser signals go dark. After synchronisation, the dark laser shots should line up
with the charge dropouts at lag 0.

In [ ]:
MAX_SHIFT = 5     # in shots; a larger offset is treated as a bad stamp and the reading dropped


def synchronise(table):
    out = table.copy()
    row = np.arange(len(table))
    moved = {}
    for p in params:
        cols = param_cols[p]
        if p == TARGET or not cols:
            continue
        if p.startswith(SLOW):
            out[cols] = table[cols].ffill()
            continue
        shift = np.round(table[f'{p}.stamp_offset'].to_numpy() / dt)
        ok = np.isfinite(shift) & (np.abs(shift) <= MAX_SHIFT)
        target_row = row + np.where(ok, shift, 0).astype(int)
        ok &= (target_row >= 0) & (target_row < len(table))
        values = table[cols].to_numpy(float)
        new = np.full_like(values, np.nan)
        new[target_row[ok]] = values[ok]        # a stale reading lands back on its own shot
        out[cols] = new
        moved[p] = int((shift[ok] != 0).sum())
    return out, moved


synced, moved = synchronise(recorded)
print('readings moved to another shot:', {p: n for p, n in moved.items() if n})


def dropout_lag(table, col):
    # for lags -3..3: how many charge dropouts coincide with a dark shot of `col`
    x = table[col].abs()
    dark = x < DROP_FRAC * x.rolling(201, center=True, min_periods=20).median()
    agree = {k: int((table['is_dropout'] & dark.shift(-k, fill_value=False)).sum())
             for k in range(-3, 4)}
    return max(agree, key=agree.get), agree


laser_cols = [c for c in ('BTV0125.intensity', 'laser_scope.peak', 'clex_laser_power')
              if c in recorded.columns]
if not recorded['is_dropout'].any():
    print('\nThis run has no charge dropouts, so the cross-check cannot be done here. '
          'Try it on another run.')
else:
    print(f"\ndark laser shots matching the {recorded['is_dropout'].sum()} charge dropouts, per lag:")
    for col in laser_cols:
        for label, table in (('before', recorded), ('after', synced)):
            lag, agree = dropout_lag(table, col)
            if max(agree.values()) == 0:
                print(f'  {col:20s} {label:6s}: no dark shots -- this signal does not see the dropouts')
            else:
                print(f'  {col:20s} {label:6s}: best lag {lag:+d}   {agree}')

In [ ]:
if laser_cols and recorded['is_dropout'].any():
    laser_col = laser_cols[0]
    start = max(recorded.index[recorded['is_dropout']][0] - 15, recorded.index[0])
    window = slice(start, start + 30)                  # 30 shots around the first dropout
    fig, axes = plt.subplots(2, 1, figsize=(10, 4.5), sharex=True)
    for ax, table, title in ((axes[0], recorded, 'before synchronisation'),
                             (axes[1], synced, 'after synchronisation')):
        ax.plot(table.loc[window, TARGET], 'C0')
        ax2 = ax.twinx()
        ax2.plot(table.loc[window, laser_col], 'C3')
        ax.set_title(title)
        ax.grid(alpha=0.3)
        ax.set_ylabel(TARGET, color='C0')
        ax2.set_ylabel(laser_col, color='C3')
    axes[1].set_xlabel('shot')
    plt.tight_layout()
    plt.show()

**What to look for:** `best lag +0` after synchronisation, for every laser signal that sees
the dropouts. If the dips in the two curves line up only in the lower plot, synchronisation
worked.

### 3d. Waveforms: the best correlation window

`peak` and `integral` summarise a whole RF or scope trace, but often only part of the pulse
matters for the charge, e.g. the RF phase at the moment the laser hits the cathode. Each
trace is stored as 48 window averages (`.w00` … `.w47`). For each waveform, this finds the
range of windows whose average correlates best with the charge. That average becomes one
more feature, `<NAME>.win`.

Two precautions:
- It uses **the training part only** (shots before the test region), so the test region
  stays unseen.
- Searching many windows always finds *some* correlation, even in pure noise. So the search
  is repeated on the charge **circularly shifted** by a random large offset, which keeps
  each signal's own slow drift but breaks any real shot-to-shot link. The best of these
  shifted searches is the **noise floor**. A window becomes a feature only if it beats it.

In [ ]:
MAX_WIN = 6       # longest window range, in windows
N_NULL = 20       # circular shifts for the noise floor: beating the largest of 20 ~ p < 0.05


def noise_floor(stat, x, y, k=N_NULL, seed=SEED):
    # largest value of stat(x, y shifted) over k circular shifts of y
    rng = np.random.default_rng(seed)
    n = len(y)
    shifts = rng.integers(n // 10, n - n // 10, size=k) if n >= 20 else range(1, k + 1)
    return max(stat(x, np.roll(y, int(s))) for s in shifts)


def best_window(windows_matrix, y):
    # (|r|, first, last+1) of the window range whose average correlates best with y
    best = (0.0, 0, 1)
    n_win = windows_matrix.shape[1]
    for a in range(n_win):
        for b in range(a + 1, min(a + MAX_WIN, n_win) + 1):
            x = windows_matrix[:, a:b].mean(1)
            r = abs(np.corrcoef(x, y)[0, 1]) if x.std() > 0 else 0.0
            if r > best[0]:
                best = (r, a, b)
    return best


CUT = cut_shot(recorded)     # first shot of the test region -- the same for every team
train_part = synced.loc[(synced.index < CUT) & ~synced['is_dropout']]

best_windows, profiles, win_features = {}, {}, {}
for p in params:
    wcols = [c for c in param_cols[p] if is_window(c)]
    if not wcols:
        continue
    B, y = train_part[wcols].to_numpy(float), train_part[TARGET].to_numpy(float)
    ok = np.isfinite(B).all(1) & np.isfinite(y)
    if ok.sum() < 50:
        continue
    B, y = B[ok], y[ok]
    profiles[p] = [abs(np.corrcoef(B[:, i], y)[0, 1]) if B[:, i].std() > 0 else 0
                   for i in range(len(wcols))]
    r, a, b = best_window(B, y)
    floor = noise_floor(lambda B_, y_: best_window(B_, y_)[0], B, y)
    best_windows[p] = wcols[a:b]
    if r > floor:                                   # only real windows become features
        win_features[f'{p}.win'] = synced[best_windows[p]].mean(axis=1)
    verdict = 'signal' if r > floor else 'no better than noise'
    print(f'{p:14s} best windows {wcols[a][-3:]}..{wcols[b - 1][-3:]}   |r| = {r:.3f}   '
          f'noise floor {floor:.3f}   -> {verdict}')
synced = pd.concat([synced, pd.DataFrame(win_features)], axis=1)

if profiles:
    fig, ax = plt.subplots(figsize=(10, 3))
    for p, prof in profiles.items():
        ax.plot(prof, label=p)
    ax.set_xlabel('window along the trace')
    ax.set_ylabel(f'|r| with {TARGET}')
    ax.set_title('Where along each waveform the charge information sits (training part)')
    ax.legend(fontsize=7, ncol=3)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

### 3e. Full correlation study

Four measures of how much each variable says about the charge, on synchronised training
shots that are not dropouts:

| Measure | Detects |
|---|---|
| Pearson | linear relations only |
| Spearman | any monotonic relation |
| Distance correlation | any dependence; zero only if the two are independent |
| Mutual information | any dependence, in nats (not bounded by 1) |

A variable with a low Pearson but high values on the other three is a non-linear driver,
one a linear model would miss.

**Not every recorded signal is informative**, and some only look informative by chance.
Each measure gets the same kind of noise floor as in 3d. A variable counts as
**informative** only if it beats the floor on at least two of the four measures. "No better
than noise" is a finding, not a failure.

In [ ]:
def distance_corr(x, y):
    # Szekely distance correlation (uses n^2 memory, so it runs on a subsample)
    def centred(a):
        d = np.abs(a[:, None] - a[None, :])
        return d - d.mean(0) - d.mean(1)[:, None] + d.mean()
    A, B = centred(x), centred(y)
    dcov = (A * B).mean()
    return np.sqrt(max(dcov, 0) / np.sqrt((A * A).mean() * (B * B).mean()))


MEASURES = {
    'pearson': lambda x, y: abs(np.corrcoef(x, y)[0, 1]),
    'spearman': lambda x, y: abs(spearmanr(x, y).statistic),
    'distance': distance_corr,
    'mutual_info': lambda x, y: mutual_info_regression(x[:, None], y, random_state=SEED)[0],
}
SUBSAMPLE = {'distance': 1000, 'mutual_info': 5000}    # the two slow measures use fewer shots
N_NULL_MEASURE = {'mutual_info': 10}                     # and MI uses fewer shifts
MIN_MEASURES = 2      # informative = beats its noise floor on at least 2 of the 4 measures

not_features = {'t', 'pc_time', 'is_dropout', 'is_bad_device'}
candidates = [c for c in synced.columns if c not in not_features and c != TARGET
              and not is_window(c) and not c.endswith('_offset')]
# training part only: choosing features on the test region would leak it into the results
clean = synced.loc[(synced.index < CUT) & ~synced['is_dropout'] & ~synced['is_bad_device']]
clean = clean.dropna(subset=[TARGET])
features = [c for c in candidates if clean[c].notna().mean() > 0.5 and clean[c].nunique() > 1]

rows = []
for c in features:
    pair = clean[[c, TARGET]].dropna()
    x, y = pair[c].to_numpy(float), pair[TARGET].to_numpy(float)
    row = {'feature': c}
    for name, f in MEASURES.items():
        # a contiguous block of shots, not random ones, so the circular shifts keep their meaning
        k = SUBSAMPLE.get(name, len(x))
        row[name] = f(x[:k], y[:k])
        row[f'{name}_floor'] = noise_floor(f, x[:k], y[:k], k=N_NULL_MEASURE.get(name, N_NULL))
    rows.append(row)

study = pd.DataFrame(rows).set_index('feature').sort_values('pearson', ascending=False)
names = list(MEASURES)
beats = pd.DataFrame({n: study[n] > study[f'{n}_floor'] for n in names})
study['informative'] = beats.sum(axis=1) >= MIN_MEASURES
corr = study[names]

skipped = sorted(set(candidates) - set(features))
print(f'{len(features)} features scored. Constant or mostly missing, skipped: {skipped}')
print(f"{study['informative'].sum()} of {len(study)} beat the noise floor on >= {MIN_MEASURES} "
      f"measures. No better than noise: {list(study.index[~study['informative']])}")
study[names + ['informative']].round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 0.28 * len(corr) + 1.5))
ax.imshow(corr.to_numpy() / corr.max().to_numpy(), cmap='Blues', aspect='auto', vmin=0, vmax=1)
for i in range(corr.shape[0]):
    for j in range(corr.shape[1]):
        ax.text(j, i, f'{corr.iat[i, j]:.2f}', ha='center', va='center', fontsize=7)
ax.set_xticks(range(corr.shape[1]))
ax.set_xticklabels(corr.columns)
ax.set_yticks(range(len(corr)))
ax.set_yticklabels(corr.index, fontsize=8)
ax.set_title(f'How much each feature says about {TARGET}\n(colour normalised per column)')
plt.tight_layout()
plt.show()

**Redundancy within each family.** Many variables are near-copies of each other: two
readbacks of the same RF amplitude, or a camera feature and a power meter seeing the same
laser fluctuation. Within each family, the variable most related to the charge is kept,
and anything correlated with a kept one above `REDUNDANT` is dropped. The result,
`selected`, is a starting feature list for the Medium and Hard tasks (section 4).

In [ ]:
FAMILIES = {
    'RF':      ('PG', 'PK'),
    'LASER':   ('laser_scope', 'lab_laser', 'clex_laser', 'BTV'),
    'MAGNETS': ('SNH', 'SNI', 'DHG'),
    'AMBIENT': ('TEMP', 'HUMID'),
}
REDUNDANT = 0.95      # |correlation| above which two features count as copies


def family(c):
    return next((f for f, prefixes in FAMILIES.items() if c.startswith(prefixes)), 'OTHER')


informative = study.index[study['informative']]
if len(informative) < 3:
    print('!! Fewer than 3 features beat the noise floor -- using all of them instead.')
    informative = study.index
ranked = corr.loc[informative]                 # already sorted by |pearson|

selected = []
fams = sorted({family(c) for c in ranked.index})
fig, axes = plt.subplots(1, len(fams), figsize=(4.5 * len(fams), 4.5), squeeze=False)
for ax, fam in zip(axes[0], fams):
    cols = [c for c in ranked.index if family(c) == fam]
    C = clean[cols].corr().abs()
    keep = []
    for c in cols:
        if all(C.loc[c, k] < REDUNDANT for k in keep):
            keep.append(c)
    selected += keep
    ax.imshow(C, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(len(cols)))
    ax.set_xticklabels(cols, rotation=90, fontsize=6)
    ax.set_yticks(range(len(cols)))
    ax.set_yticklabels([c if c in keep else f'({c})' for c in cols], fontsize=6)
    ax.set_title(f'{fam}: keep {len(keep)} of {len(cols)}', fontsize=9)
plt.tight_layout()
plt.show()
print(f'{len(selected)} selected features (dropped ones are in brackets above):')
print(selected)

> **Your turn (Easy, see `objectives.md`)**
> - Reduce every run you use (2c), and work out its regime from the data: stable, saturated, single or multi-bunch,
>   laser scan? Does it change within the run?
> - Check the lag of every fast parameter separately, not only the laser.
> - Go beyond the ready-made features: better waveform features, better camera features.
> - Give a reason for every variable you keep or drop, and check whether your ranking holds
>   across runs.

## 4. 🟡 Medium and 🔴 Hard — over to you

From here on there is no worked solution: building the models is the challenge. The goals
are in `objectives.md`.

- **🟡 Medium — virtual diagnostic:** infer the charge of shot `t` from the other variables
  of **the same shot**. Then add the charge of past shots as inputs, and see how much it
  helps.
- **🔴 Hard — forecasting:** predict the charge of shot `t + h` from everything known up to
  shot `t`: one step ahead, then several.

**What you have from section 3:**

| Name | What it is |
|---|---|
| `synced` | the synchronised table, one row per shot, with the flags `is_dropout` and `is_bad_device` |
| `selected` | the features kept by the correlation study |
| `CUT` | the first shot of the test region: train only on shots before it |
| `recorded` | the table as recorded: pass it to the evaluation helpers of 1b |

**Evaluating your models** with the helpers of 1b:
- virtual diagnostic: `score_vd(recorded, pred)`, where `pred` is a Series of predictions
  indexed by shot;
- forecasting: `score_forecast(recorded, pred)`, where `pred` is a DataFrame with columns
  `shot`, `h`, `pred`, and `shot` is predicted from information up to `shot − h`. It also
  gives `dMAE` against the rolling mean.

The cell below computes the simplest references, so you know where you start.

In [ ]:
test_shots = scored_shots(recorded)            # test shots that are not dropouts

# Medium: the simplest virtual diagnostic predicts the mean training charge for every shot
train_mean = synced.loc[(synced.index < CUT) & ~synced['is_dropout'], TARGET].mean()
print('Medium, simplest reference (the training mean):')
print('  ', score_vd(recorded, pd.Series(train_mean, index=test_shots)))

# Hard: the rolling mean of the last W good shots, the forecast to beat, at several horizons
baseline = rolling_baseline(recorded)
HORIZONS = [h for h in (1, 2, 5, 10, 20, 50) if h <= CUT // 4]   # short runs: fewer horizons
naive = pd.concat([pd.DataFrame({'shot': test_shots, 'h': h,
                                 'pred': baseline.reindex(test_shots - h).to_numpy()})
                   for h in HORIZONS])
print(f'\nHard, the rolling mean of the last {W} good shots, by horizon h:')
score_forecast(recorded, naive)[['MAE', 'R2', 'coverage']].round(4)

> **Your turn (Medium, see `objectives.md`)**
> - Compare a linear model, a tree ensemble and a neural network, and show what drives each
>   one. Do they agree with your correlation study?
> - Add the past charge (`BCM_GUN` of shots `< t`) as inputs: how do the results change
>   with more or less history?
> - Train on one run and test on another (e.g. single → multi-bunch, stable → laser scan).
> - Choose the metric that makes sense, and explain it physically.

> **Your turn (Hard, see `objectives.md`)**
> - Beat the rolling mean, one step ahead and then up to 50 shots ahead.
> - Try at least one model built for sequences: TFT, DeepAR, LSTM/GRU or a time-series
>   foundation model.
> - Tune the context window and show its effect.
> - Split by the shot being *predicted*: no training target may lie at or after `CUT`.

**Saving your predictions.** Keep them as CSV files in `predictions/`, one per run and
task, so they can be re-evaluated and compared later:
- virtual diagnostic: `shot,pred`
- forecasting: `shot,h,pred`

**Logging to W&B** (see `GRADING.md`), for example:

```python
import wandb
wandb.init(project='aissai-hackathon-2026',
           config={'challenge': 'optimisation-forecasting', 'team': '<your-team>',
                   'tier': 'medium', 'dataset': run_name})
wandb.log({'forecast/vd_mae': <your MAE>})
wandb.finish()
```

## 5. Next steps

- **Reduce every run you want to use**, not just one chunk, and compare: which results hold
  and which depend on the run's regime?
- **Improve the features of 2c.** The best window of 3d is a start; a fit to the pulse
  shape, or features learned from the raw traces and images, may do better.
- **Optional:**
  - split the charge into frequency bands (FFT) and forecast slow drift and fast jitter
    separately;
  - prediction intervals, with their coverage checked on the test region;
  - close the loop: use your forecaster inside one of the parent challenge's optimisers
    to choose laser or RF settings that keep the forecast charge on target.